In [7]:
import os
import re
import time
from pathlib import Path
from collections import OrderedDict
import xml.etree.ElementTree as ET

# ── Configuration ───────────────────────────────────────────
# ── Local Configuration ───────────────────────────────────────
# Place the source XML files directly in your active project workspace folder
GREEK_XML  = "/Users/gcrane/github/canonical-greekLit/data/tlg0016/tlg001/tlg0016.tlg001.perseus-grc2.xml"
ENG_XML    = "/Users/gcrane/github/canonical-greekLit/data/tlg0016/tlg001/tlg0016.tlg001.perseus-eng2.xml"

# Direct output targeting to a standard folder relative to the notebook
OUT_DIR    = Path("/Users/gcrane/Downloads/gemsite/perseus_site3")
OUT_DIR.mkdir(parents=True, exist_ok=True)

NS = {'tei': 'http://www.tei-c.org/ns/1.0'}
BOOK_NAMES = {
    '1': 'Clio', '2': 'Euterpe', '3': 'Thalia', '4': 'Melpomene',
    '5': 'Terpsichore', '6': 'Erato', '7': 'Polymnia', '8': 'Urania', '9': 'Calliope'
}

# ── Enhanced Perseus 4 (Hopper) CSS ─────────────────────────
PERSEUS_CSS = r"""
* { margin: 0; padding: 0; box-sizing: border-box; }
body {
    font-family: "Palatino Linotype", "Book Antiqua", Palatino, Georgia, serif;
    font-size: 13px;
    color: #000;
    background: #fff;
}
a { color: #336699; text-decoration: none; }
a:hover { text-decoration: underline; }

/* Header Banner & Collections Bar */
#perseus-banner { background: #660000; color: #fff; padding: 8px 15px; display: flex; justify-content: space-between; align-items: center; border-bottom: 2px solid #330000; }
#perseus-banner h1 { font-size: 18px; font-weight: bold; }
#perseus-banner h1 a { color: #fff; }
#perseus-banner .doc-title { font-size: 12px; color: #ffccaa; font-weight: bold; }
#nav-bar { background: #ddddcc; border-bottom: 1px solid #999988; padding: 5px 15px; font-size: 11px; color: #555; }
#nav-bar a { margin-right: 15px; color: #003366; font-weight: bold; }

/* Visual Browse Bar (Top Layout) */
#browse-bar { background: #eeeeee; border-bottom: 1px solid #cccccc; padding: 8px 15px; font-size: 11px; }
.browse-row { margin-bottom: 4px; display: flex; align-items: flex-start; }
.browse-row label { font-weight: bold; width: 60px; color: #666; shrink: 0; }
.browse-items { display: flex; flex-wrap: wrap; gap: 4px; }
.browse-items a { padding: 1px 4px; color: #003366; border: 1px solid transparent; }
.browse-items a:hover { background: #fff; border: 1px solid #aaa; text-decoration: none; }
.browse-items a.current { background: #660000; color: #fff !important; font-weight: bold; border-radius: 2px; }

/* Framing Layout: Left TOC Sidebar + Main Columns */
#outer-wrapper { display: flex; min-height: calc(100vh - 80px); }
#sidebar-toc { width: 220px; background: #f5f5ee; border-right: 1px solid #ccccbb; padding: 12px; font-size: 11px; overflow-y: auto; }
#sidebar-toc h3 { font-size: 12px; color: #660000; margin-bottom: 8px; border-bottom: 1px solid #ccccbb; padding-bottom: 3px; }
#sidebar-toc ul { list-style: none; padding-left: 0; }
#sidebar-toc li { margin-bottom: 4px; }
#sidebar-toc li.active-book { font-weight: bold; }
#sidebar-toc .toc-chapters { padding-left: 10px; margin-top: 2px; font-weight: normal; }
#sidebar-toc .toc-chapters a.current { color: #660000; font-weight: bold; }

#main-container { flex: 1; display: flex; padding: 15px; gap: 15px; }
#focus-text { flex: 3; min-width: 0; }
#cross-ref { flex: 2; min-width: 0; border-left: 2px solid #eeeeee; padding-left: 15px; }
.panel-header { background: #660000; color: #fff; padding: 3px 8px; font-size: 11px; font-weight: bold; margin-bottom: 10px; }

/* Typography & Marginal Citation Rules */
.nav-arrows { margin: 5px 0; display: flex; align-items: center; gap: 10px; font-size: 12px; font-weight: bold; }
.nav-arrows a { font-size: 16px; color: #660000; }
.section-block { margin-bottom: 14px; position: relative; padding-left: 45px; line-height: 1.6; scroll-margin-top: 40px; }
.section-block .sec-num { position: absolute; left: 0; top: 0; color: #990000; font-size: 11px; font-weight: bold; width: 35px; text-align: right; }
.greek-text { font-size: 15px; line-height: 1.7; font-family: "Gentium Plus", "Athena", Palatino, serif; }
.english-text { font-size: 13px; color: #111; }
.note { font-size: 11px; color: #666; font-style: italic; background: #fafafa; padding: 0 2px; }
#credits { background: #f9f9f9; border-top: 1px solid #eee; padding: 10px; margin-top: 30px; font-size: 11px; color: #555; line-height: 1.4; }
#version-select { background: #f0f0f0; border: 1px solid #ddd; padding: 5px; margin-bottom: 10px; font-size: 11px; }
#version-select a { margin-right: 10px; font-weight: bold; }
#version-select a.active { color: #000; text-decoration: none; cursor: default; }
"""

# ── Parsing Helpers ─────────────────────────────────────────
def extract_text_recursive(elem):
    parts = []
    if elem.text:
        parts.append(elem.text)
    for child in elem:
        tag = child.tag.replace('{http://www.tei-c.org/ns/1.0}', '')
        if tag == 'note':
            note_text = extract_text_recursive(child).strip()
            if note_text:
                parts.append(f'<span class="note">[{note_text}]</span>')
        elif tag in ('milestone', 'reg'):
            pass
        else:
            parts.append(extract_text_recursive(child))
        if child.tail:
            parts.append(child.tail)
    return ''.join(parts)

def parse_tei(path):
    tree = ET.parse(path)
    root = tree.getroot()
    title  = (root.findtext('.//tei:titleStmt/tei:title', namespaces=NS) or '').strip()
    author = (root.findtext('.//tei:titleStmt/tei:author', namespaces=NS) or '').strip()
    editor_el = root.find('.//tei:titleStmt/tei:editor', NS)
    editor = (editor_el.text or '').strip() if editor_el is not None else ''

    body = root.find('.//tei:body', NS)
    top_div = body.find('tei:div', NS)

    data = OrderedDict()
    for book_div in top_div.findall('tei:div[@type="textpart"]', NS):
        book_n = book_div.get('n')
        data[book_n] = OrderedDict()
        for chap_div in book_div.findall('tei:div[@type="textpart"]', NS):
            chap_n = chap_div.get('n')
            data[book_n][chap_n] = OrderedDict()
            for sec_div in chap_div.findall('tei:div[@type="textpart"]', NS):
                sec_n = sec_div.get('n')
                paragraphs = sec_div.findall('tei:p', NS)
                html_parts = []
                for p in paragraphs:
                    html_parts.append(extract_text_recursive(p).strip())
                data[book_n][chap_n][sec_n] = ' '.join(html_parts)
    return data, {'title': title, 'author': author, 'editor': editor}

print("Parsing XML files...")
grc_data, grc_meta = parse_tei(GREEK_XML)
eng_data, eng_meta = parse_tei(ENG_XML)

def page_filename(book, chapter, focus='greek'):
    return f"{focus}_book{book}_ch{chapter}.html"

# ── Component Generation ────────────────────────────────────
def render_sections(sections_dict, css_class='greek-text', section_prefix=''):
    html = []
    for sec_n, text in sections_dict.items():
        label = sec_n if sec_n not in ('0', 'pr') else 'pr'
        # Assign IDs to each block for deep linking from the top panel
        html.append(
            f'<div id="{section_prefix}sec{sec_n}" class="section-block {css_class}">'
            f'<span class="sec-num">[{label}]</span>'
            f'<p>{text}</p>'
            f'</div>'
        )
    return '\n'.join(html)

def render_top_browse_bar(book_n, chap_n, focus, data):
    """Generates the stacked Visual Browsing panel matching Hopper's layout."""
    # Row 1: Books
    book_links = []
    for bn in data:
        first_ch = list(data[bn].keys())[0]
        cls = ' class="current"' if bn == book_n else ''
        book_links.append(f'<a href="{page_filename(bn, first_ch, focus)}"{cls}>{bn}</a>')
    
    # Row 2: Chapters within current Book
    chap_links = []
    for cn in data[book_n]:
        cls = ' class="current"' if cn == chap_n else ''
        chap_links.append(f'<a href="{page_filename(book_n, cn, focus)}"{cls}>{cn}</a>')
        
    # Row 3: Sections within current Chapter (Fragments linked directly inline)
    sec_links = []
    for sn in data[book_n][chap_n]:
        label = sn if sn not in ('0', 'pr') else 'pr'
        sec_links.append(f'<a href="#{focus}sec{sn}">{label}</a>')

    muse_name = BOOK_NAMES.get(book_n, "")
    return f"""<div id="browse-bar">
    <div class="browse-row">
        <label>Book:</label>
        <div class="browse-items">{' '.join(book_links)}</div>
    </div>
    <div class="browse-row">
        <label>Chapter:</label>
        <div class="browse-items">{' '.join(chap_links)}</div>
    </div>
    <div class="browse-row">
        <label>Section:</label>
        <div class="browse-items">{' '.join(sec_links)}</div>
    </div>
</div>"""

def render_sidebar_toc(current_book, current_chap, focus, data):
    """Generates the static hierarchical left-hand Table of Contents."""
    html = ["<div id=\"sidebar-toc\">", "<h3>Table of Contents</h3>", "<ul>"]
    for bn in data:
        muse = f" ({BOOK_NAMES[bn]})" if bn in BOOK_NAMES else ""
        if bn == current_book:
            html.append(f'<li class="active-book">Book {bn}{muse}')
            html.append('<ul class="toc-chapters">')
            for cn in data[bn]:
                cls = ' class="current"' if cn == current_chap else ''
                html.append(f'<li><a href="{page_filename(bn, cn, focus)}"{cls}>Chapter {cn}</a></li>')
            html.append('</ul></li>')
        else:
            first_ch = list(data[bn].keys())[0]
            html.append(f'<li><a href="{page_filename(bn, first_ch, focus)}">Book {bn}{muse}</a></li>')
    html.append("</ul></div>")
    return "\n".join(html)

# ── Main Compilation Cycle ──────────────────────────────────
chapter_list = []
for bn in grc_data:
    for cn in grc_data[bn]:
        chapter_list.append((bn, cn))

print(f"Compiling {len(chapter_list) * 2} structural site pages...")
start = time.time()

for idx, (book_n, chap_n) in enumerate(chapter_list):
    for focus in ('greek', 'english'):
        # Contextual Paging Controls
        prev_link, next_link = '', ''
        if idx > 0:
            pb, pc = chapter_list[idx - 1]
            prev_link = f'<a href="{page_filename(pb, pc, focus)}">&#x25C0;</a>'
        if idx < len(chapter_list) - 1:
            nb, nc = chapter_list[idx + 1]
            next_link = f'<a href="{page_filename(nb, nc, focus)}">&#x25B6;</a>'

        location_str = f"Hdt. {book_n}.{chap_n}"
        nav_html = f'<div class="nav-arrows">{prev_link} <span class="location">{location_str}</span> {next_link}</div>'

        # Orientation Layout assignments
        if focus == 'greek':
            f_data, c_data = grc_data, eng_data
            f_class, c_class = 'greek-text', 'english-text'
            f_label, c_label = f"Herodotus, Histories (Greek Text)", f"English Translation (tr. {eng_meta['editor']})"
            other_focus, f_ed = 'english', f"ed. {grc_meta['editor']}"
        else:
            f_data, c_data = eng_data, grc_data
            f_class, c_class = 'english-text', 'greek-text'
            f_label, c_label = f"Herodotus, Histories (English Translation)", f"Greek text (ed. {grc_meta['editor']})"
            other_focus, f_ed = 'greek', f"tr. {eng_meta['editor']}"

        focus_html = render_sections(f_data[book_n][chap_n], f_class, focus)
        cross_html = render_sections(c_data.get(book_n, {}).get(chap_n, {}), c_class, other_focus)

        browse_bar = render_top_browse_bar(book_n, chap_n, focus, f_data)
        sidebar_toc = render_sidebar_toc(book_n, chap_n, focus, f_data)
        book_display = f"Book {book_n}" + (f" ({BOOK_NAMES[book_n]})" if book_n in BOOK_NAMES else '')

        # HTML Framework Synthesis
        page_html = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>Herodotus, Histories, {book_display}, chapter {chap_n}</title>
  <style>{PERSEUS_CSS}</style>
</head>
<body>

<div id="perseus-banner">
  <h1><a href="{page_filename('1', '1', 'greek')}">Perseus Digital Library</a></h1>
  <span class="doc-title">{f_label}</span>
</div>

<div id="nav-bar">
  <a href="{page_filename('1', '1', 'greek')}">Home</a>
  <a href="{page_filename(book_n, chap_n, other_focus)}">Configure Side-by-Side Focus ({other_focus.title()})</a>
</div>

{browse_bar}

<div id="outer-wrapper">
  {sidebar_toc}
  
  <div id="main-container">
    <div id="focus-text">
      <div id="version-select">
        <strong>Focus Version:</strong>
        <a href="{page_filename(book_n, chap_n, 'greek')}" class="{'active' if focus=='greek' else ''}">Greek (Godley)</a>
        <a href="{page_filename(book_n, chap_n, 'english')}" class="{'active' if focus=='english' else ''}">English (Godley)</a>
      </div>

      {nav_html}
      <h2 style="font-size:14px; color:#660000; margin: 10px 0; border-bottom: 1px dotted #ccc; padding-bottom:3px;">
        {book_display}, Chapter {chap_n}
      </h2>

      {focus_html}
      {nav_html}

      <div id="credits">
        <p><strong>Herodotus.</strong> <em>The Histories</em>, {f_ed}. Cambridge, Harvard University Press.</p>
        <p>Text provided by the Perseus Digital Library under Creative Commons terms.</p>
      </div>
    </div>

    <div id="cross-ref">
      <div class="panel-header">{c_label}</div>
      {cross_html}
    </div>
  </div>
</div>

</body>
</html>"""
        
        fname = page_filename(book_n, chap_n, focus)
        (OUT_DIR / fname).write_text(page_html, encoding='utf-8')

print(f"Success! Finished compiling site pages in {time.time() - start:.2f} seconds.")

Parsing XML files...
Compiling 3156 structural site pages...
Success! Finished compiling site pages in 1.52 seconds.


In [5]:
import os
import re
import time
from pathlib import Path
from collections import OrderedDict
import xml.etree.ElementTree as ET

# ── Local Configuration ───────────────────────────────────────
# Place the source XML files directly in your active project workspace folder
GREEK_XML  = "/Users/gcrane/github/canonical-greekLit/data/tlg0016/tlg001/tlg0016.tlg001.perseus-grc2.xml"
ENG_XML    = "/Users/gcrane/github/canonical-greekLit/data/tlg0016/tlg001/tlg0016.tlg001.perseus-eng2.xml"

# Direct output targeting to a standard folder relative to the notebook
OUT_DIR    = Path("/Users/gcrane/Downloads/gemsite/perseus_site2")
OUT_DIR.mkdir(parents=True, exist_ok=True)

NS = {'tei': 'http://www.tei-c.org/ns/1.0'}
BOOK_NAMES = {
    '1': 'Clio', '2': 'Euterpe', '3': 'Thalia', '4': 'Melpomene',
    '5': 'Terpsichore', '6': 'Erato', '7': 'Polymnia', '8': 'Urania', '9': 'Calliope'
}

# ── Pure HTML5/CSS Layout (Zero JavaScript) ──────────────────
PERSEUS_CSS = r"""
* { margin: 0; padding: 0; box-sizing: border-box; }
body {
    font-family: "Palatino Linotype", "Book Antiqua", Palatino, Georgia, serif;
    font-size: 13px;
    color: #000;
    background: #fff;
}
a { color: #336699; text-decoration: none; }
a:hover { text-decoration: underline; }

/* Header Banner */
#perseus-banner { background: #660000; color: #fff; padding: 8px 15px; display: flex; justify-content: space-between; align-items: center; border-bottom: 2px solid #330000; }
#perseus-banner h1 { font-size: 18px; font-weight: bold; }
#perseus-banner h1 a { color: #fff; }
#perseus-banner .doc-title { font-size: 12px; color: #ffccaa; font-weight: bold; }
#nav-bar { background: #ddddcc; border-bottom: 1px solid #999988; padding: 5px 15px; font-size: 11px; color: #555; }
#nav-bar a { margin-right: 15px; color: #003366; font-weight: bold; }

/* Hopper Visual Navigation Matrix */
#browse-bar { background: #eeeeee; border-bottom: 1px solid #cccccc; padding: 8px 15px; font-size: 11px; }
.browse-row { margin-bottom: 4px; display: flex; align-items: flex-start; }
.browse-row label { font-weight: bold; width: 60px; color: #666; flex-shrink: 0; }
.browse-items { display: flex; flex-wrap: wrap; gap: 4px; }
.browse-items a { padding: 1px 4px; color: #003366; border: 1px solid transparent; }
.browse-items a:hover { background: #fff; border: 1px solid #aaa; text-decoration: none; }
.browse-items a.current { background: #660000; color: #fff !important; font-weight: bold; border-radius: 2px; }

/* Framework Framing Panels */
#outer-wrapper { display: flex; min-height: calc(100vh - 80px); }
#sidebar-toc { width: 220px; background: #f5f5ee; border-right: 1px solid #ccccbb; padding: 12px; font-size: 11px; overflow-y: auto; }
#sidebar-toc h3 { font-size: 12px; color: #660000; margin-bottom: 8px; border-bottom: 1px solid #ccccbb; padding-bottom: 3px; }
#sidebar-toc ul { list-style: none; padding-left: 0; }
#sidebar-toc li { margin-bottom: 4px; }
#sidebar-toc li.active-book { font-weight: bold; }
#sidebar-toc .toc-chapters { padding-left: 10px; margin-top: 2px; font-weight: normal; }
#sidebar-toc .toc-chapters a.current { color: #660000; font-weight: bold; }

#main-container { flex: 1; display: flex; padding: 15px; gap: 15px; }
#focus-text { flex: 3; min-width: 0; }
#cross-ref { flex: 2; min-width: 0; border-left: 2px solid #eeeeee; padding-left: 15px; }
.panel-header { background: #660000; color: #fff; padding: 3px 8px; font-size: 11px; font-weight: bold; margin-bottom: 10px; }

/* Navigation and Citation Anchors */
.nav-arrows-group { border-bottom: 1px solid #ddd; padding-bottom: 6px; margin-bottom: 10px; }
.nav-arrows { display: flex; align-items: center; gap: 8px; font-size: 12px; font-weight: bold; margin-bottom: 4px; }
.nav-arrows a { font-size: 14px; color: #660000; }
.nav-arrows .nav-label { color: #555; font-size: 11px; font-weight: normal; width: 110px; }

/* Content Blocks & Fragment Offset Positioning */
.section-block { margin-bottom: 14px; position: relative; padding-left: 45px; line-height: 1.6; scroll-margin-top: 60px; }
.section-block:target { background-color: #ffffcc; border-radius: 2px; } /* Highlights targeted section */
.section-block .sec-num { position: absolute; left: 0; top: 0; color: #990000; font-size: 11px; font-weight: bold; width: 35px; text-align: right; }
.greek-text { font-size: 16px; line-height: 1.7; font-family: "Gentium Plus", "Athena", Palatino, serif; }
.english-text { font-size: 13px; color: #111; }
.note { font-size: 11px; color: #666; font-style: italic; background: #fafafa; padding: 0 2px; }
#credits { background: #f9f9f9; border-top: 1px solid #eee; padding: 10px; margin-top: 30px; font-size: 11px; color: #555; }
#version-select { background: #f0f0f0; border: 1px solid #ddd; padding: 5px; margin-bottom: 10px; font-size: 11px; }
#version-select a { margin-right: 10px; font-weight: bold; }
#version-select a.active { color: #000; text-decoration: none; cursor: default; }
"""

# ── XML Parsing Initialization ──────────────────────────────
def extract_text_recursive(elem):
    parts = []
    if elem.text: parts.append(elem.text)
    for child in elem:
        tag = child.tag.replace('{http://www.tei-c.org/ns/1.0}', '')
        if tag == 'note':
            note_text = extract_text_recursive(child).strip()
            if note_text: parts.append(f'<span class="note">[{note_text}]</span>')
        elif tag in ('milestone', 'reg'): pass
        else: parts.append(extract_text_recursive(child))
        if child.tail: parts.append(child.tail)
    return ''.join(parts)

def parse_tei(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Source file missing: {os.path.abspath(path)}")
    tree = ET.parse(path)
    root = tree.getroot()
    body = root.find('.//tei:body', NS)
    top_div = body.find('tei:div', NS)

    data = OrderedDict()
    for book_div in top_div.findall('tei:div[@type="textpart"]', NS):
        book_n = book_div.get('n')
        data[book_n] = OrderedDict()
        for chap_div in book_div.findall('tei:div[@type="textpart"]', NS):
            chap_n = chap_div.get('n')
            data[book_n][chap_n] = OrderedDict()
            for sec_div in chap_div.findall('tei:div[@type="textpart"]', NS):
                sec_n = sec_div.get('n')
                paragraphs = sec_div.findall('tei:p', NS)
                html_parts = []
                for p in paragraphs:
                    html_parts.append(extract_text_recursive(p).strip())
                data[book_n][chap_n][sec_n] = ' '.join(html_parts)
    return data

print("Loading and compiling source TEI XML text parts...")
try:
    grc_data = parse_tei(GREEK_XML)
    eng_data = parse_tei(ENG_XML)
except FileNotFoundError as e:
    print(f"\n[CRITICAL ERROR] {e}\nPlease check file names or placements in step 1.")
    raise

def page_filename(book, chapter, focus='greek'):
    return f"{focus}_book{book}_ch{chapter}.html"

def render_sections(sections_dict, css_class='greek-text'):
    html = []
    for sec_n, text in sections_dict.items():
        label = sec_n if sec_n not in ('0', 'pr') else 'pr'
        html.append(
            f'<div id="sec{sec_n}" class="section-block {css_class}">'
            f'<span class="sec-num">[{label}]</span>'
            f'<p>{text}</p>'
            f'</div>'
        )
    return '\n'.join(html)

def render_top_browse_bar(book_n, chap_n, focus, data):
    book_links = []
    for bn in data:
        first_ch = list(data[bn].keys())[0]
        cls = ' class="current"' if bn == book_n else ''
        book_links.append(f'<a href="{page_filename(bn, first_ch, focus)}"{cls}>{bn}</a>')
    
    chap_links = []
    for cn in data[book_n]:
        cls = ' class="current"' if cn == chap_n else ''
        chap_links.append(f'<a href="{page_filename(book_n, cn, focus)}"{cls}>{cn}</a>')
        
    sec_links = []
    for sn in data[book_n][chap_n]:
        label = sn if sn not in ('0', 'pr') else 'pr'
        sec_links.append(f'<a href="#sec{sn}">{label}</a>')

    return f"""<div id="browse-bar">
    <div class="browse-row"><label>Book:</label><div class="browse-items">{' '.join(book_links)}</div></div>
    <div class="browse-row"><label>Chapter:</label><div class="browse-items">{' '.join(chap_links)}</div></div>
    <div class="browse-row"><label>Section:</label><div class="browse-items">{' '.join(sec_links)}</div></div>
</div>"""

def render_sidebar_toc(current_book, current_chap, focus, data):
    html = ["<div id=\"sidebar-toc\">", "<h3>Table of Contents</h3>", "<ul>"]
    for bn in data:
        muse = f" ({BOOK_NAMES[bn]})" if bn in BOOK_NAMES else ""
        if bn == current_book:
            html.append(f'<li class="active-book">Book {bn}{muse} <ul class="toc-chapters">')
            for cn in data[bn]:
                cls = ' class="current"' if cn == current_chap else ''
                html.append(f'<li><a href="{page_filename(bn, cn, focus)}"{cls}>Chapter {cn}</a></li>')
            html.append('</ul></li>')
        else:
            first_ch = list(data[bn].keys())[0]
            html.append(f'<li><a href="{page_filename(bn, first_ch, focus)}">Book {bn}{muse}</a></li>')
    html.append("</ul></div>")
    return "\n".join(html)

# ── Flat Index Assembly ───────────────────────────────────────
chapter_list = []
section_map = OrderedDict() # Keeps a sequential path across all sections globally

for bn in grc_data:
    for cn in grc_data[bn]:
        chapter_list.append((bn, cn))
        for sn in grc_data[bn][cn]:
            section_map[(bn, cn, sn)] = True

sec_list = list(section_map.keys())

print(f"Generating site framework files...")
start_time = time.time()
pages_count = 0

for ch_idx, (book_n, chap_n) in enumerate(chapter_list):
    for focus in ('greek', 'english'):
        
        # 1. Chapter Level Paging Rules (e.g., 1.1 -> 1.2)
        ch_prev_html, ch_next_html = '&#x25C0;', '&#x25B6;'
        if ch_idx > 0:
            pb, pc = chapter_list[ch_idx - 1]
            ch_prev_html = f'<a href="{page_filename(pb, pc, focus)}" title="Go to Chapter {pb}.{pc}">&#x25C0;</a>'
        if ch_idx < len(chapter_list) - 1:
            nb, nc = chapter_list[ch_idx + 1]
            ch_next_html = f'<a href="{page_filename(nb, nc, focus)}" title="Go to Chapter {nb}.{nc}">&#x25B6;</a>'

        # 2. Section Level Paging Rules (e.g., 1.1.1 -> 1.1.2)
        # Find the primary section inside this specific chapter scope
        first_sec_in_ch = list(grc_data[book_n][chap_n].keys())[0]
        
        # Pull global positional indicators
        current_sec_global_idx = sec_list.index((book_n, chap_n, first_sec_in_ch))
        
        # Previous Section logic
        if current_sec_global_idx > 0:
            ps_b, ps_c, ps_s = sec_list[current_sec_global_idx - 1]
            sec_prev_html = f'<a href="{page_filename(ps_b, ps_c, focus)}#sec{ps_s}" title="Go to Section {ps_b}.{ps_c}.{ps_s}">&#x25C0;</a>'
        else:
            sec_prev_html = '&#x25C0;'
            
        # Next Section logic (jump past all local sections to track boundary jumps)
        total_local_sections = len(grc_data[book_n][chap_n])
        next_sec_global_idx = current_sec_global_idx + total_local_sections
        
        if next_sec_global_idx < len(sec_list):
            ns_b, ns_c, ns_s = sec_list[next_sec_global_idx]
            sec_next_html = f'<a href="{page_filename(ns_b, ns_c, focus)}#sec{ns_s}" title="Go to Section {ns_b}.{ns_c}.{ns_s}">&#x25B6;</a>'
        else:
            sec_next_html = '&#x25B6;'

        # Building Navigation Control Elements
        nav_html = f"""<div class="nav-arrows-group">
            <div class="nav-arrows">
                {ch_prev_html} <span class="nav-label">Chapter Navigation:</span> {ch_next_html} 
                <span style="color:#666; font-size:11px; margin-left:10px;">Hdt. {book_n}.{chap_n}</span>
            </div>
            <div class="nav-arrows">
                {sec_prev_html} <span class="nav-label">Section Navigation:</span> {sec_next_html}
            </div>
        </div>"""

        focus_html = render_sections(grc_data[book_n][chap_n], 'greek-text' if focus=='greek' else 'english-text')
        cross_html = render_sections(eng_data[book_n][chap_n], 'english-text' if focus=='greek' else 'greek-text')

        browse_bar = render_top_browse_bar(book_n, chap_n, focus, grc_data)
        sidebar_toc = render_sidebar_toc(book_n, chap_n, focus, grc_data)
        book_display = f"Book {book_n} ({BOOK_NAMES.get(book_n, '')})"
        other_focus = 'english' if focus == 'greek' else 'greek'

        page_content = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>Herodotus, Histories, {book_display}, Chapter {chap_n}</title>
  <style>{PERSEUS_CSS}</style>
</head>
<body>

<div id="perseus-banner">
  <h1><a href="{page_filename('1', '1', 'greek')}">Perseus Digital Library</a></h1>
  <span class="doc-title">Herodotus, Histories ({focus.title()} Focus)</span>
</div>

<div id="nav-bar">
  <a href="{page_filename('1', '1', 'greek')}">Home</a>
  <a href="{page_filename(book_n, chap_n, other_focus)}">Switch to {other_focus.title()} Focus</a>
</div>

{browse_bar}

<div id="outer-wrapper">
  {sidebar_toc}
  
  <div id="main-container">
    <div id="focus-text">
      <div id="version-select">
        <strong>Focus Context:</strong>
        <a href="{page_filename(book_n, chap_n, 'greek')}" class="{'active' if focus=='greek' else ''}">Greek (Godley)</a>
        <a href="{page_filename(book_n, chap_n, 'english')}" class="{'active' if focus=='english' else ''}">English (Godley)</a>
      </div>

      {nav_html}
      <h2 style="font-size:14px; color:#660000; margin:10px 0; border-bottom:1px solid #ccc; padding-bottom:3px;">
        {book_display}, Chapter {chap_n}
      </h2>

      {focus_html}
      {nav_html}

      <div id="credits">
        <p><strong>Herodotus.</strong> <em>The Histories</em>, ed. A. D. Godley. Cambridge, Harvard University Press.</p>
      </div>
    </div>

    <div id="cross-ref">
      <div class="panel-header">Cross Reference Translation</div>
      {cross_html}
    </div>
  </div>
</div>

</body>
</html>"""
        
        fname = page_filename(book_n, chap_n, focus)
        (OUT_DIR / fname).write_text(page_content, encoding='utf-8')
        pages_count += 1

print(f"Success! Compiled {pages_count} interconnected pages in {time.time() - start_time:.2f} seconds.")

Loading and compiling source TEI XML text parts...
Generating site framework files...
Success! Compiled 3156 interconnected pages in 1.67 seconds.


In [10]:
import os
import re
import time
from pathlib import Path
from collections import OrderedDict
import xml.etree.ElementTree as ET

# ── Configuration ───────────────────────────────────────────
GREEK_XML  = "/Users/gcrane/github/canonical-greekLit/data/tlg0016/tlg001/tlg0016.tlg001.perseus-grc2.xml"
ENG_XML    = "/Users/gcrane/github/canonical-greekLit/data/tlg0016/tlg001/tlg0016.tlg001.perseus-eng2.xml"
OUT_DIR    = Path("/Users/gcrane/Downloads/gemsite/perseus_site4")
OUT_DIR.mkdir(parents=True, exist_ok=True)

NS = {'tei': 'http://www.tei-c.org/ns/1.0'}
BOOK_NAMES = {
    '1': 'Clio', '2': 'Euterpe', '3': 'Thalia', '4': 'Melpomene',
    '5': 'Terpsichore', '6': 'Erato', '7': 'Polymnia', '8': 'Urania', '9': 'Calliope'
}

# ── Authentic Hopper Layout CSS (Zero Javascript) ───────────
PERSEUS_CSS = r"""
* { margin: 0; padding: 0; box-sizing: border-box; }
body {
    font-family: "Palatino Linotype", "Book Antiqua", Palatino, Georgia, serif;
    font-size: 13px;
    color: #000;
    background: #fff;
}
a { color: #336699; text-decoration: none; }
a:hover { text-decoration: underline; }

#perseus-banner { background: #660000; color: #fff; padding: 8px 15px; display: flex; justify-content: space-between; align-items: center; border-bottom: 2px solid #330000; }
#perseus-banner h1 { font-size: 18px; font-weight: bold; }
#perseus-banner h1 a { color: #fff; }
#perseus-banner .doc-title { font-size: 12px; color: #ffccaa; font-weight: bold; }
#nav-bar { background: #ddddcc; border-bottom: 1px solid #999988; padding: 5px 15px; font-size: 11px; color: #555; }
#nav-bar a { margin-right: 15px; color: #003366; font-weight: bold; }

/* Dynamic Visual Navigation Rows (Hopper Tile Grid Style) */
#browse-bar { background: #eeeeee; border-bottom: 1px solid #cccccc; padding: 8px 15px; font-size: 11px; }
.browse-row { margin-bottom: 6px; display: flex; align-items: center; }
.browse-row label { font-weight: bold; width: 80px; color: #555; flex-shrink: 0; }
.browse-items { display: flex; flex-wrap: wrap; gap: 3px; align-items: center; }

/* Proportional Size Tiles matching Perseus 4 Interface */
.nav-tile-wrapper { display: inline-flex; flex-direction: column; align-items: center; text-align: center; }
.nav-tile-link { display: block; padding: 2px 6px; background: #e0e0d0; color: #336699; border: 1px solid #bbbb99; min-width: 24px; text-align: center; font-weight: bold; }
.nav-tile-link:hover { background: #fff; text-decoration: none; border-color: #660000; }
.nav-tile-link.current { background: #660000; color: #fff !important; border-color: #330000; }
.size-bar { height: 3px; background: #990000; margin-top: 2px; border-radius: 1px; min-width: 4px; }

/* Content Framing Panels */
#outer-wrapper { display: flex; min-height: calc(100vh - 80px); }
#sidebar-toc { width: 240px; background: #f5f5ee; border-right: 1px solid #ccccbb; padding: 12px; font-size: 11px; overflow-y: auto; flex-shrink: 0; }
#sidebar-toc h3 { font-size: 12px; color: #660000; margin-bottom: 8px; border-bottom: 1px solid #ccccbb; padding-bottom: 3px; }
#sidebar-toc ul { list-style: none; padding-left: 0; }
#sidebar-toc li { margin-bottom: 5px; }
#sidebar-toc li.active-book { font-weight: bold; color: #660000; }

/* Multi-Tier Sidebar Nesting Layout */
.toc-chapters { padding-left: 10px; margin-top: 3px; font-weight: normal; }
.toc-chapters li { margin-bottom: 3px; }
.toc-chapters a { color: #003366; }
.toc-chapters a.current { color: #660000; font-weight: bold; }
.toc-sections { padding-left: 12px; margin-top: 2px; display: flex; flex-wrap: wrap; gap: 4px; font-weight: normal; }
.toc-sections a { background: #e0e0d4; padding: 1px 4px; border-radius: 2px; color: #333; font-size: 10px; }
.toc-sections a:hover { background: #660000; color: #fff; text-decoration: none; }
.toc-sections a.current { background: #660000; color: #fff !important; font-weight: bold; }

#main-container { flex: 1; display: flex; padding: 15px; gap: 15px; }
#focus-text { flex: 3; min-width: 0; }
#cross-ref { flex: 2; min-width: 0; border-left: 2px solid #eeeeee; padding-left: 15px; }
.panel-header { background: #660000; color: #fff; padding: 3px 8px; font-size: 11px; font-weight: bold; margin-bottom: 10px; }

.tier-indicator { background: #fffccb; border: 1px solid #e6db55; padding: 4px 8px; font-size: 11px; margin-bottom: 10px; font-weight: bold; color: #555; display: inline-block; }
.nav-arrows { display: flex; align-items: center; gap: 10px; font-size: 12px; font-weight: bold; margin: 8px 0; border-bottom: 1px dotted #ccc; padding-bottom: 6px; }
.nav-arrows a { font-size: 14px; color: #660000; }

.section-block { margin-bottom: 14px; position: relative; padding-left: 45px; line-height: 1.6; }
.section-block .sec-num { position: absolute; left: 0; top: 0; color: #990000; font-size: 11px; font-weight: bold; width: 35px; text-align: right; }
.section-block .sec-num a { color: #990000; text-decoration: underline; }
.greek-text { font-size: 16px; line-height: 1.7; font-family: "Gentium Plus", "Athena", Palatino, serif; }
.english-text { font-size: 13px; color: #111; }
.note { font-size: 11px; color: #666; font-style: italic; background: #fafafa; }
#credits { background: #f9f9f9; border-top: 1px solid #eee; padding: 10px; margin-top: 30px; font-size: 11px; color: #555; }
#version-select { background: #f0f0f0; border: 1px solid #ddd; padding: 5px; margin-bottom: 10px; font-size: 11px; }
#version-select a { margin-right: 10px; font-weight: bold; }
#version-select a.active { color: #000; text-decoration: none; cursor: default; }
"""

# ── XML Parsing Helper ──────────────────────────────────────
def extract_text_recursive(elem):
    parts = []
    if elem.text: parts.append(elem.text)
    for child in elem:
        tag = child.tag.replace('{http://www.tei-c.org/ns/1.0}', '')
        if tag == 'note':
            note_text = extract_text_recursive(child).strip()
            if note_text: parts.append(f'<span class="note">[{note_text}]</span>')
        elif tag in ('milestone', 'reg'): pass
        else: parts.append(extract_text_recursive(child))
        if child.tail: parts.append(child.tail)
    return ''.join(parts)

def parse_tei(path):
    tree = ET.parse(path)
    root = tree.getroot()
    body = root.find('.//tei:body', NS)
    top_div = body.find('tei:div', NS)

    data = OrderedDict()
    for book_div in top_div.findall('tei:div[@type="textpart"]', NS):
        book_n = book_div.get('n')
        data[book_n] = OrderedDict()
        for chap_div in book_div.findall('tei:div[@type="textpart"]', NS):
            chap_n = chap_div.get('n')
            data[book_n][chap_n] = OrderedDict()
            for sec_div in chap_div.findall('tei:div[@type="textpart"]', NS):
                sec_n = sec_div.get('n')
                paragraphs = sec_div.findall('tei:p', NS)
                html_parts = []
                for p in paragraphs:
                    html_parts.append(extract_text_recursive(p).strip())
                data[book_n][chap_n][sec_n] = ' '.join(html_parts)
    return data

print("Parsing XML records...")
grc_data = parse_tei(GREEK_XML)
eng_data = parse_tei(ENG_XML)

# ── Routing Filename Layouts ────────────────────────────────
def chapter_filename(book, chapter, focus='greek'):
    return f"{focus}_book{book}_ch{chapter}.html"

def section_filename(book, chapter, section, focus='greek'):
    return f"{focus}_book{book}_ch{chapter}_sec{section}.html"

# ── Proportional Size Estimation Analytics ──────────────────
# Calculate character volume sizing metrics across the database to calibrate the visual bars
chapter_sizes = {}
for bn in grc_data:
    for cn in grc_data[bn]:
        # Sum text characters in this chapter block context
        total_chars = sum(len(txt) for txt in grc_data[bn][cn].values())
        chapter_sizes[(bn, cn)] = total_chars

max_char_len = max(chapter_sizes.values()) if chapter_sizes else 1

def calculate_visual_width(book, chapter, max_px=30):
    """Calculates relative horizontal pixels for indicator bar."""
    size = chapter_sizes.get((book, chapter), 0)
    ratio = size / max_char_len
    return max(4, int(ratio * max_px))

# ── Component Building Blocks ───────────────────────────────
def render_sections_block(book_n, chap_n, sections_dict, focus, is_isolated=False, isolated_sec_n=None):
    html = []
    for sec_n, text in sections_dict.items():
        if is_isolated and sec_n != isolated_sec_n:
            continue
        label = sec_n if sec_n not in ('0', 'pr') else 'pr'
        css_class = 'greek-text' if focus == 'greek' else 'english-text'
        
        sec_file_link = section_filename(book_n, chap_n, sec_n, focus)
        html.append(
            f'<div class="section-block {css_class}">'
            f'<span class="sec-num"><a href="{sec_file_link}" title="Isolate Section File">{label}</a></span>'
            f'<p>{text}</p>'
            f'</div>'
        )
    return '\n'.join(html)

def render_top_navigation_matrix(book_n, chap_n, focus, data, routing_tier='chapter'):
    """Top Navigation Grid containing proportional layout bars."""
    # Row 1: Books
    book_links = []
    for bn in data:
        first_ch = list(data[bn].keys())[0]
        cls = ' class="current"' if bn == book_n else ''
        dest = chapter_filename(bn, first_ch, focus) if routing_tier == 'chapter' else section_filename(bn, first_ch, list(data[bn][first_ch].keys())[0], focus)
        book_links.append(f'<a href="{dest}"{cls} style="font-weight:bold; padding:2px 6px; margin-right:4px;">Book {bn}</a>')
    
    # Row 2: Chapters featuring relative structural bar charts (Perseus 4 Style)
    chap_links = []
    for cn in data[book_n]:
        cls = ' current' if cn == chap_n else ''
        dest = chapter_filename(book_n, cn, focus) if routing_tier == 'chapter' else section_filename(book_n, cn, list(data[book_n][cn].keys())[0], focus)
        
        # Determine bar thickness relative to content metrics
        pixel_width = calculate_visual_width(book_n, cn)
        
        chap_links.append(
            f'<div class="nav-tile-wrapper">'
            f'  <a href="{dest}" class="nav-tile-link{cls}">{cn}</a>'
            f'  <div class="size-bar" style="width:{pixel_width}px;" title="Relative Context Length"></div>'
            f'</div>'
        )
        
    return f"""<div id="browse-bar">
    <div class="browse-row"><label>Books:</label><div class="browse-items">{' '.join(book_links)}</div></div>
    <div class="browse-row" style="margin-top:8px;"><label>Chapters:</label><div class="browse-items" style="gap:5px 3px;">{' '.join(chap_links)}</div></div>
</div>"""

def render_sidebar_toc(current_book, current_chap, current_sec_n, focus, data, routing_tier='chapter'):
    """Sidebar Layout tracking deep section structural expansions dynamically."""
    html = ["<div id=\"sidebar-toc\">", "<h3>Table of Contents</h3>", "<ul>"]
    for bn in data:
        muse = f" ({BOOK_NAMES[bn]})" if bn in BOOK_NAMES else ""
        if bn == current_book:
            html.append(f'<li class="active-book">Book {bn}{muse} <ul class="toc-chapters">')
            for cn in data[bn]:
                is_active_ch = (cn == current_chap)
                cls = ' class="current"' if is_active_ch else ''
                dest = chapter_filename(bn, cn, focus) if routing_tier == 'chapter' else section_filename(bn, cn, list(data[bn][cn].keys())[0], focus)
                html.append(f'<li><a href="{dest}"{cls}>Chapter {cn}</a>')
                
                # DEEP ROUTE INNER NESTING: Expand sections exclusively for the active target chapter list
                if is_active_ch:
                    html.append('<ul class="toc-sections">')
                    for sn in data[bn][cn]:
                        label = sn if sn not in ('0', 'pr') else 'pr'
                        s_cls = ' class="current"' if (routing_tier == 'section' and sn == current_sec_n) else ''
                        s_dest = section_filename(bn, cn, sn, focus)
                        html.append(f'<li><a href="{s_dest}"{s_cls}>{label}</a></li>')
                    html.append('</ul>')
                
                html.append('</li>')
            html.append('</ul></li>')
        else:
            first_ch = list(data[bn].keys())[0]
            dest = chapter_filename(bn, first_ch, focus) if routing_tier == 'chapter' else section_filename(bn, first_ch, list(data[bn][first_ch].keys())[0], focus)
            html.append(f'<li><a href="{dest}">Book {bn}{muse}</a></li>')
    html.append("</ul></div>")
    return "\n".join(html)

# ── Sequenced Flat Arrays Extraction ─────────────────────────
chapter_list = []
section_list = []

for bn in grc_data:
    for cn in grc_data[bn]:
        chapter_list.append((bn, cn))
        for sn in grc_data[bn][cn]:
            section_list.append((bn, cn, sn))

print("Compiling dynamic graphical structural asset views...")
start_time = time.time()
total_compiled = 0

# ==========================================
# PHASE 1: GENERATE CHAPTER-LEVEL SEPARATE FILES
# ==========================================
for ch_idx, (book_n, chap_n) in enumerate(chapter_list):
    for focus in ('greek', 'english'):
        prev_html, next_html = '&#x25C0;', '&#x25B6;'
        if ch_idx > 0:
            pb, pc = chapter_list[ch_idx - 1]
            prev_html = f'<a href="{chapter_filename(pb, pc, focus)}">&#x25C0; Prev Chapter ({pb}.{pc})</a>'
        if ch_idx < len(chapter_list) - 1:
            nb, nc = chapter_list[ch_idx + 1]
            next_html = f'<a href="{chapter_filename(nb, nc, focus)}">Next Chapter ({nb}.{nc}) &#x25B6;</a>'
            
        nav_html = f'<div class="nav-arrows">{prev_html} <span style="margin:0 auto;color:#666;">Viewing Chapter Level</span> {next_html}</div>'

        focus_html = render_sections_block(book_n, chap_n, grc_data[book_n][chap_n], focus)
        cross_html = render_sections_block(book_n, chap_n, eng_data[book_n][chap_n], 'english' if focus == 'greek' else 'greek')

        browse_bar = render_top_navigation_matrix(book_n, chap_n, focus, grc_data, routing_tier='chapter')
        sidebar_toc = render_sidebar_toc(book_n, chap_n, None, focus, grc_data, routing_tier='chapter')
        other_focus = 'english' if focus == 'greek' else 'greek'
        
        page_content = f"""<!DOCTYPE html>
<html>
<head><meta charset="UTF-8"><title>Hdt. Chapter {book_n}.{chap_n}</title><style>{PERSEUS_CSS}</style></head>
<body>
<div id="perseus-banner"><h1><a href="{chapter_filename('1', '1', 'greek')}">Perseus Digital Library</a></h1><span class="doc-title">Chapter Route Mode</span></div>
<div id="nav-bar"><a href="{chapter_filename('1', '1', 'greek')}">Home</a> <a href="{chapter_filename(book_n, chap_n, other_focus)}">Switch Focus to {other_focus.title()}</a></div>
{browse_bar}
<div id="outer-wrapper">
  {sidebar_toc}
  <div id="main-container">
    <div id="focus-text">
      <div class="tier-indicator">Standard Chapter Context View (Click section index tag to isolate)</div>
      {nav_html}
      <h2 style="font-size:14px;color:#660000;margin-bottom:10px;">Book {book_n}, Chapter {chap_n}</h2>
      {focus_html}
      {nav_html}
    </div>
    <div id="cross-ref"><div class="panel-header">Translation Alignment</div>{cross_html}</div>
  </div>
</div>
</body>
</html>"""
        (OUT_DIR / chapter_filename(book_n, chap_n, focus)).write_text(page_content, encoding='utf-8')
        total_compiled += 1

# ==========================================
# PHASE 2: GENERATE SECTION-LEVEL SEPARATE FILES
# ==========================================
for sec_idx, (book_n, chap_n, sec_n) in enumerate(section_list):
    for focus in ('greek', 'english'):
        prev_html, next_html = '&#x25C0;', '&#x25B6;'
        if sec_idx > 0:
            pb, pc, ps = section_list[sec_idx - 1]
            prev_html = f'<a href="{section_filename(pb, pc, ps, focus)}">&#x25C0; Prev Section ({pb}.{pc}.{ps})</a>'
        if sec_idx < len(section_list) - 1:
            nb, nc, ns = section_list[sec_idx + 1]
            next_html = f'<a href="{section_filename(nb, nc, ns, focus)}">Next Section ({nb}.{nc}.{ns}) &#x25B6;</a>'
            
        nav_html = f'<div class="nav-arrows">{prev_html} <span style="margin:0 auto;color:#666;">Viewing Section Level</span> {next_html}</div>'

        focus_html = render_sections_block(book_n, chap_n, grc_data[book_n][chap_n], focus, is_isolated=True, isolated_sec_n=sec_n)
        cross_html = render_sections_block(book_n, chap_n, eng_data[book_n][chap_n], 'english' if focus == 'greek' else 'greek', is_isolated=True, isolated_sec_n=sec_n)

        browse_bar = render_top_navigation_matrix(book_n, chap_n, focus, grc_data, routing_tier='section')
        sidebar_toc = render_sidebar_toc(book_n, chap_n, sec_n, focus, grc_data, routing_tier='section')
        other_focus = 'english' if focus == 'greek' else 'greek'
        
        page_content = f"""<!DOCTYPE html>
<html>
<head><meta charset="UTF-8"><title>Hdt. Section {book_n}.{chap_n}.{sec_n}</title><style>{PERSEUS_CSS}</style></head>
<body>
<div id="perseus-banner"><h1><a href="{chapter_filename('1', '1', 'greek')}">Perseus Digital Library</a></h1><span class="doc-title">Section Route Mode</span></div>
<div id="nav-bar"><a href="{chapter_filename('1', '1', 'greek')}">Home</a> <a href="{section_filename(book_n, chap_n, sec_n, other_focus)}">Switch Focus to {other_focus.title()}</a> | <a href="{chapter_filename(book_n, chap_n, focus)}" style="color:#660000;">&uarr; Up to Chapter View</a></div>
{browse_bar}
<div id="outer-wrapper">
  {sidebar_toc}
  <div id="main-container">
    <div id="focus-text">
      <div class="tier-indicator" style="background:#e3f2fd; border-color:#90caf9;">Isolated Section Context View</div>
      {nav_html}
      <h2 style="font-size:14px;color:#660000;margin-bottom:10px;">Book {book_n}, Chapter {chap_n}, Section {sec_n}</h2>
      {focus_html}
      {nav_html}
    </div>
    <div id="cross-ref"><div class="panel-header">Translation Alignment</div>{cross_html}</div>
  </div>
</div>
</body>
</html>"""
        (OUT_DIR / section_filename(book_n, chap_n, sec_n, focus)).write_text(page_content, encoding='utf-8')
        total_compiled += 1

print(f"Compilation finished. Generated {total_compiled} independent static HTML view files in {time.time() - start_time:.2f} seconds.")

Parsing XML records...
Compiling dynamic graphical structural asset views...
Compilation finished. Generated 11832 independent static HTML view files in 10.10 seconds.
